# Phase 4: NLP Threat Intelligence with HuggingFace Transformers

**Objective:** Extend Aegis AI beyond numerical telemetry by applying transformer-based NLP to unstructured threat data — security advisories, CVE descriptions, and IDS alert logs.

**Why this matters:** Real-world SOC analysts deal with both structured logs and unstructured text (threat reports, CVE bulletins, Slack alerts). This module bridges the gap using HuggingFace's `transformers` library.

**Pipeline:**
1. Zero-shot threat classification with `facebook/bart-large-mnli`
2. Semantic threat similarity search using `sentence-transformers`
3. Named Entity Recognition (NER) for extracting IPs, CVEs, and attack vectors
4. Fine-tuning DistilBERT on a custom threat label dataset

In [ ]:
# Install dependencies (run once)
# !pip install transformers sentence-transformers datasets torch accelerate -q

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

device = 0 if torch.cuda.is_available() else -1
print(f'HuggingFace device: {"GPU" if device == 0 else "CPU"}')
print(f'PyTorch version: {torch.__version__}')

## 1. Threat Log Dataset
Realistic IDS alert messages and CVE-style descriptions across five threat categories.

In [ ]:
# Representative threat intelligence corpus
THREAT_CORPUS = [
    # DoS / DDoS
    ("Massive SYN flood detected from 192.168.1.0/24 — 50,000 packets/sec toward port 443",             "DoS"),
    ("UDP amplification attack using open DNS resolvers causing service degradation",                    "DoS"),
    ("HTTP GET flood targeting /api/login endpoint, 200k requests in 60 seconds",                       "DoS"),
    ("ICMP ping flood from multiple spoofed source IPs; bandwidth saturation at 98%",                  "DoS"),
    ("Slowloris attack holding 10,000 partial HTTP connections to Apache web server",                   "DoS"),

    # Reconnaissance / Probe
    ("Nmap SYN scan detected across 65535 ports from external IP 203.0.113.42",                       "Probe"),
    ("SNMP bulk walk attempt against network devices — possible topology enumeration",                  "Probe"),
    ("DNS zone transfer request from unauthorized host — potential network mapping",                    "Probe"),
    ("Aggressive OS fingerprinting via TCP/IP stack analysis detected by IDS sensor",                  "Probe"),
    ("Banner grabbing on FTP, SSH, SMTP services from single source over 5 minutes",                   "Probe"),

    # Remote-to-Local (R2L)
    ("Multiple failed SSH login attempts with common username/password combinations — brute force",    "R2L"),
    ("CVE-2023-44487 HTTP/2 Rapid Reset attack exploited against load balancer",                       "R2L"),
    ("FTP anonymous write access abused to upload malicious PHP webshell",                             "R2L"),
    ("Credential stuffing attack using leaked database against /api/auth endpoint",                    "R2L"),
    ("Exploitation of CVE-2021-44228 Log4Shell in Apache Log4j for remote code execution",            "R2L"),

    # User-to-Root (U2R)
    ("Suspicious sudo -l enumeration followed by kernel exploit attempt CVE-2023-0386",               "U2R"),
    ("SUID binary abuse detected: /usr/bin/find executed with shell escalation payload",              "U2R"),
    ("Container escape via runc vulnerability — process moved from container to host namespace",       "U2R"),
    ("Dirty COW (CVE-2016-5195) privilege escalation attempt on unpatched Linux kernel",              "U2R"),
    ("Exploitation of misconfigured sudoers file allowing www-data to run /bin/bash as root",        "U2R"),

    # Advanced Persistent Threat (APT) / Exfiltration
    ("Low-and-slow beaconing to C2 server at 10-minute intervals — possible APT implant",            "APT"),
    ("DNS tunneling detected: unusually long TXT record queries to unknown external domain",          "APT"),
    ("Large data exfiltration over encrypted HTTPS to cloud storage not in whitelist",               "APT"),
    ("Living-off-the-land: PowerShell executing encoded commands after phishing email open",         "APT"),
    ("Lateral movement via SMB pass-the-hash across internal subnet 10.0.0.0/16",                   "APT"),
]

df_threats = pd.DataFrame(THREAT_CORPUS, columns=['text', 'label'])
print(df_threats['label'].value_counts())
df_threats.head()

## 2. Zero-Shot Threat Classification (No Training Required)

In [ ]:
# Load zero-shot classifier from HuggingFace Hub
print('Loading facebook/bart-large-mnli ...')
zsc = pipeline('zero-shot-classification',
               model='facebook/bart-large-mnli',
               device=device)

CANDIDATE_LABELS = ['DoS attack', 'network probe', 'remote exploit', 'privilege escalation', 'APT exfiltration']
LABEL_MAP = {
    'DoS attack': 'DoS',
    'network probe': 'Probe',
    'remote exploit': 'R2L',
    'privilege escalation': 'U2R',
    'APT exfiltration': 'APT'
}

# Classify test alerts
test_alerts = [
    "Massive SYN flood from botnet targeting port 80",
    "Nmap aggressive scan across all TCP ports from 203.0.113.5",
    "SSH brute force using rockyou.txt wordlist — 10k attempts in 1 hour",
    "Kernel exploit CVE-2023-0386 used to escalate from www-data to root",
    "Slow DNS beaconing to command-and-control server every 15 minutes"
]

print('\n=== Zero-Shot Classification Results ===')
results = []
for alert in test_alerts:
    out = zsc(alert, candidate_labels=CANDIDATE_LABELS)
    predicted = LABEL_MAP[out['labels'][0]]
    confidence = out['scores'][0]
    results.append({'Alert': alert[:60] + '...', 'Predicted': predicted, 'Confidence': f'{confidence:.2%}'})
    print(f"  Alert : {alert[:60]}")
    print(f"  Label : {predicted} ({confidence:.2%} confidence)\n")

pd.DataFrame(results)

## 3. Semantic Threat Similarity Search

In [ ]:
print('Loading sentence-transformers/all-MiniLM-L6-v2 ...')
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Encode all threat corpus
corpus_texts = df_threats['text'].tolist()
corpus_embeddings = embedder.encode(corpus_texts, convert_to_tensor=True, show_progress_bar=True)

# Query: find most similar known threats for a new alert
query = "Beaconing traffic to suspicious external IP at regular 8-minute intervals"
query_embedding = embedder.encode(query, convert_to_tensor=True)

hits = util.semantic_search(query_embedding, corpus_embeddings, top_k=3)[0]

print(f'\nQuery: "{query}"')
print('\nTop-3 Similar Known Threats:')
for i, hit in enumerate(hits, 1):
    idx = hit['corpus_id']
    print(f"  {i}. [{df_threats.iloc[idx]['label']}] Score: {hit['score']:.4f}")
    print(f"     {corpus_texts[idx]}")

In [ ]:
# Visualise inter-class similarity matrix
sim_matrix = util.cos_sim(corpus_embeddings, corpus_embeddings).numpy()
labels_short = [f"{row['label'][:3]}-{i%5+1}" for i, row in df_threats.iterrows()]

plt.figure(figsize=(12, 10))
sns.heatmap(sim_matrix, xticklabels=labels_short, yticklabels=labels_short,
            cmap='RdYlGn', vmin=0, vmax=1, annot=False)
plt.title('Semantic Similarity Matrix — Threat Intelligence Corpus', fontsize=13)
plt.tight_layout()
plt.savefig('../src/results/threat_similarity_matrix.png', dpi=150)
plt.show()
print('Saved: src/results/threat_similarity_matrix.png')

## 4. Named Entity Recognition — Extract IPs, CVEs, Attack Vectors

In [ ]:
import re

def extract_threat_entities(text):
    """Regex-based NER for structured threat intelligence extraction."""
    entities = {
        'IPs':     re.findall(r'\b(?:\d{1,3}\.){3}\d{1,3}(?:/\d{1,2})?\b', text),
        'CVEs':    re.findall(r'CVE-\d{4}-\d{4,7}', text, re.IGNORECASE),
        'Ports':   re.findall(r'port\s+(\d{1,5})', text, re.IGNORECASE),
        'Protocols': [p for p in ['SSH','HTTP','HTTPS','FTP','DNS','SMB','UDP','TCP','ICMP','SNMP']
                      if re.search(r'\b' + p + r'\b', text, re.IGNORECASE)],
    }
    return entities

print('=== Threat Entity Extraction ===')
samples = [
    "Exploitation of CVE-2021-44228 Log4Shell in Apache Log4j for remote code execution from 203.0.113.42 via HTTP port 8080",
    "SSH brute force from 192.168.1.100 targeting 10.0.0.5 on port 22 — 5,000 attempts",
    "DNS tunneling via UDP port 53 to C2 at 198.51.100.7; CVE-2023-44487 also observed"
]
for s in samples:
    ents = extract_threat_entities(s)
    print(f'\nAlert: {s[:70]}...')
    for k, v in ents.items():
        if v: print(f'  {k}: {v}')

## 5. Fine-Tune DistilBERT on Threat Classification

In [ ]:
from datasets import Dataset
from transformers import DataCollatorWithPadding

# Encode labels
LABEL2ID = {'DoS': 0, 'Probe': 1, 'R2L': 2, 'U2R': 3, 'APT': 4}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

df_threats['label_id'] = df_threats['label'].map(LABEL2ID)

# Load DistilBERT tokenizer
MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding=True, max_length=128)

hf_dataset = Dataset.from_pandas(
    df_threats[['text', 'label_id']].rename(columns={'label_id': 'labels'})
)
split = hf_dataset.train_test_split(test_size=0.2, seed=42)
tokenized = split.map(tokenize, batched=True)

print(f'Train samples: {len(tokenized["train"])}')
print(f'Test  samples: {len(tokenized["test"])}')

In [ ]:
# Load DistilBERT for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=5,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

training_args = TrainingArguments(
    output_dir='../src/models/distilbert_threat',
    num_train_epochs=10,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    logging_steps=5,
    report_to='none'   # disable wandb
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['test'],
    tokenizer=tokenizer,
    data_collator=data_collator
)

print('Fine-tuning DistilBERT on threat classification ...')
trainer.train()
print('Fine-tuning complete.')

In [ ]:
# Evaluate and show predictions
preds_output = trainer.predict(tokenized['test'])
y_pred = np.argmax(preds_output.predictions, axis=1)
y_true = preds_output.label_ids

print('=== Fine-Tuned DistilBERT — Classification Report ===')
print(classification_report(y_true, y_pred, target_names=list(LABEL2ID.keys()), zero_division=0))

# Save model for inference
model.save_pretrained('../src/models/distilbert_threat')
tokenizer.save_pretrained('../src/models/distilbert_threat')
print('Model saved to src/models/distilbert_threat/')

## 6. End-to-End Inference Pipeline

In [ ]:
# Unified inference: classify any new alert with confidence scores
classifier = pipeline(
    'text-classification',
    model='../src/models/distilbert_threat',
    tokenizer='../src/models/distilbert_threat',
    return_all_scores=True,
    device=device
)

new_alerts = [
    "Repeated failed authentication from Tor exit node — possible credential stuffing",
    "High-volume ICMP traffic causing network congestion across multiple VLANs",
    "Suspicious outbound connection to known APT C2 infrastructure — possible data theft"
]

print('=== Aegis AI — NLP Threat Intelligence Inference ===')
for alert in new_alerts:
    scores = classifier(alert)[0]
    top = max(scores, key=lambda x: x['score'])
    print(f"\n  Alert     : {alert}")
    print(f"  Prediction: {top['label']} ({top['score']:.2%} confidence)")
    print(f"  All scores: { {s['label']: f\"{s['score']:.2%}\" for s in scores} }")

print('\nPhase 4 Complete: HuggingFace transformer pipeline integrated into Aegis AI.')